## Computational Physics -- Problem Set 10
##### _Theodore Landa_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')

In [ ]:
rng = np.random.default_rng(0)

### Problem 1.

_(a)_ Wrote the program!

_(b)_ I got `0.027956`, which is pretty close to 1 / 36!

In [ ]:
def roll2(N):
    return rng.integers(low=1, high=7, size=(N, 2))

print(roll2(1))

In [ ]:
bigroll2 = roll2(1_000_000)
print(np.sum((np.sum(bigroll2, axis=1) == 12)) / 1_000_000)
print(1 / 36)

### Problem 2.

Check out my plots! The first one is the four numbers as a function of time on the same axes, and the second is just the Tl isotope, because it looks like it's zero (but isn't) on the scale of the main plot.

In [ ]:
minute = 60
h_Bi = 46 * minute
p_Bi = 1 - 2**(-1/h_Bi)
h_Tl = 2.2 * minute
p_Tl = 1 - 2**(-1/h_Tl)
h_Pb = 3.3 * minute
p_Pb = 1 - 2**(-1/h_Pb)
p_Bi_Tl = 0.0209

In [ ]:
def decay(counts_input):
    N_Bi_213, N_Tl, N_Pb, N_Bi_209 = counts_input
    Pb_decay = np.sum(rng.random(N_Pb) < p_Pb)
    Tl_decay = np.sum(rng.random(N_Tl) < p_Tl)
    Bi_decay = np.sum(rng.random(N_Bi_213) < p_Bi)
    Bi_Tl = np.sum(rng.random(Bi_decay) < p_Bi_Tl)
    N_Bi_209 = N_Bi_209 + Pb_decay
    N_Pb = N_Pb - Pb_decay + Tl_decay + Bi_decay - Bi_Tl
    N_Tl = N_Tl - Tl_decay + Bi_Tl
    N_Bi_213 = N_Bi_213 - Bi_decay
    return np.array([N_Bi_213, N_Tl, N_Pb, N_Bi_209])

In [ ]:
times = np.arange(20_000)
counts = np.zeros((20_000, 4), dtype="int64")
counts[0] = np.array([10_000, 0, 0, 0])

for i in range(20_000 - 1):
    counts[i + 1] = decay(counts[i])

In [ ]:
plt.plot(times, counts[:,0], label="Bi-213 count")
plt.plot(times, counts[:,1], label="Tl count")
plt.plot(times, counts[:,2], label="Pb count")
plt.plot(times, counts[:,3], label="Bi-209 count")
plt.xlabel("time t [s]")
plt.ylabel("isotope count")
plt.title("Radioactive Decay Chain of Bi-213")
plt.legend()
plt.show()

In [ ]:
plt.plot(times, counts[:,1], label="Tl count", c="C1")
plt.xlabel("time t [s]")
plt.ylabel("isotope count")
plt.title("Tl count")
plt.show()

### Problem 3.

Check out my plots! The first one is actually loglog (and the two points Hogg asked for are invisible because they're both zero). The second one is loglog with a linear cutoff in y, so you can see all the points. Also, the second two points have zero uncertainty, hence the lack of bars. I don't know much about the curse of dimensionality, but I assume that in this context, it's referring to the fact that in high dimensional spaces, there's an enormous amount of volume, and so points become very far apart in the metric, and volumes of objects with finite extents become pretty small in comparison to the total space.

In [ ]:
def hypersphere_vol(d, N):
    V = 2**d
    expf = np.sum(np.linalg.norm(rng.random((N, d)), axis=1) <= 1) / N
    varf = expf - expf**2
    unc = V * np.sqrt(varf) / np.sqrt(N)
    return np.array([V * expf, unc])

In [ ]:
v10, unc10 = hypersphere_vol(10, 1_000_000)
v30, unc30 = hypersphere_vol(30, 1_000_000)
v100, unc100 = hypersphere_vol(100, 1_000_000)

In [ ]:
dims = np.array([10, 30, 100])
vols = np.array([v10, v30, v100])
uncs = np.array([unc10, unc30, unc100])
plt.loglog(dims, vols, 'ok', ms=2)
plt.errorbar(dims, vols, yerr=uncs, fmt="none", ecolor="red", elinewidth=1)
plt.title("Volume of a d-dimensional hypersphere (MC)")
plt.xlabel("Dimension")
plt.ylabel("Volume")
plt.show()

In [ ]:
plt.plot(dims, vols, 'ok', ms=2, label="volume")
plt.errorbar(dims, vols, yerr=uncs, fmt="none", ecolor="red", elinewidth=1, label="uncertainty")
plt.yscale('symlog', linthresh=2.8)
plt.title("Volume of a d-dimensional hypersphere (MC)")
plt.xlabel("Dimension")
plt.ylabel("Volume")
plt.ylim(-0.1, 5)
plt.legend()
plt.show()

### Problem 4.

_(a)_ My minimum is `-0.0034014539758651817`. CHeck out the plot!

_(b)_ My minimum is `15.961423878629812`. CHeck out the plot!

In [ ]:
def func(x):
  return x**2 - np.cos(4 * np.pi * x)


Tmax = 1
Tmin = 1e-3
a = 1e-4
t = 0
x_storage = []
t_storage = []
x = 2


#Find minimum with Monte Carlo BABY
#Did you know I have a monte carlo poker chip in my room
#I found it in physics and was so happy

T = Tmax
while T > Tmin:
  t +=1
  T = Tmax*np.exp(-a * t)

  if (t%10 ==0):
    t_storage.append(t)
    x_storage.append(x)

  rng = np.random.default_rng()
  delta_temp = rng.normal()
  oldx = x
  x = x + delta_temp
  change_f = func(x) - func(oldx)

  if(rng.random() > np.exp(-change_f/T)):
    x = oldx

print(x)

plt.scatter(t_storage, x_storage, color = 'red', s = 0.2)
plt.xlabel('Time Step', size = 15)
plt.title("Simulated Annealing")
plt.ylabel('X Value', size = 15)
plt.show()

In [ ]:
def func(x):
  return np.cos(x) + np.cos(np.sqrt(2) * x) + np.cos(np.sqrt(3) * x)


Tmax = 1
Tmin = 1e-4
a = 1e-3
t = 0
x_storage = []
t_storage = []
x = 10


#Find minimum with Monte Carlo BABY
#Did you know I have a monte carlo poker chip in my room
#I found it in physics and was so happy

T = Tmax
while T > Tmin:
  t +=1
  T = Tmax*np.exp(-a * t)

  if (t%10 ==0):
    t_storage.append(t)
    x_storage.append(x)

  rng = np.random.default_rng()
  delta_temp = rng.normal()
  oldx = x
  x = x + delta_temp
  change_f = func(x) - func(oldx)

  if(rng.random() > np.exp(-change_f/T)):
    x = oldx

print(x)

plt.scatter(t_storage, x_storage, color = 'red', s = 0.2)
plt.xlabel('Time Step', size = 15)
plt.ylabel('X Value', size = 15)
plt.title("Simulated Annealing", fontsize = 18)
plt.show()

### Credits.
Credit to numpy and matplotlib documentation, and to GABE ROSEN-TURITS FOR HELPING ME WITH THE LAST ONE!!